In [1]:
import pandas as pd
import numpy as np
import altair as alt

In [2]:
url = 'https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv'


In [3]:

df_ufo = pd.read_csv(
url,
names=[
    'datetime', 'city', 'state', 'country', 'shape',
    'duration_seconds', 'duration', 'notes',
    'date_posted', 'latitude', 'longitude'
],
header=None,
skiprows=1
)

df_ufo['datetime'] = pd.to_datetime(df_ufo['datetime'], errors='coerce')
df_ufo['year'] = df_ufo['datetime'].dt.year
df_ufo['month'] = df_ufo['datetime'].dt.month

df_ufo

,datetime,city,state,country,shape,duration_seconds,duration,notes,date_posted,latitude,longitude,year,month
0,1949-10-10 21:00:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082,1949,10
1,1955-10-10 17:00:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667,1955,10
2,1956-10-10 21:00:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956,10
3,1960-10-10 20:00:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960,10
4,1961-10-10 19:00:00,bristol,tn,us,sphere,300.0,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80326,2013-09-09 21:15:00,nashville,tn,us,light,600.0,10 minutes,Round from the distance/slowly changing colors...,9/30/2013,36.165833,-86.784444,2013,9
80327,2013-09-09 22:00:00,boise,id,us,circle,1200.0,20 minutes,Boise&#44 ID&#44 spherical&#44 20 min&#44 10 r...,9/30/2013,43.613611,-116.202500,2013,9
80328,2013-09-09 22:00:00,napa,ca,us,other,1200.0,hour,Napa UFO&#44,9/30/2013,38.297222,-122.284444,2013,9
80329,2013-09-09 22:20:00,vienna,va,us,circle,5.0,5 seconds,Saw a five gold lit cicular craft moving fastl...,9/30/2013,38.901111,-77.265556,2013,9


In [ ]:
df_ufo['datetime'] = pd.to_datetime(df_ufo['datetime'], errors='coerce')
df_ufo['year'] = df_ufo['datetime'].dt.year

In [5]:
# 1.  CLEANING 
df_ufo['shape'] = df_ufo['shape'].astype(str).str.strip()
trash_values = ['', 'nan', 'None', 'null', 'undefined']
df_clean = df_ufo[~df_ufo['shape'].isin(trash_values)].copy()

if hasattr(df_clean['shape'], 'cat'):
    df_clean['shape'] = df_clean['shape'].cat.remove_unused_categories()

shape_counts = df_clean['shape'].value_counts().reset_index()
shape_counts.columns = ['shape', 'count']

## Chart 1

In [6]:
# Define the hover selection
hover_select = alt.selection_point(
    on='mouseover', 
    nearest=True, 
    fields=['shape'], 
    empty=False
)

shape_counts = shape_counts[shape_counts['shape'].str.strip().str.len() > 0]

# Base Chart
base = alt.Chart(shape_counts).encode(
    y=alt.Y('shape:N', 
        title='UFO Shape', 
        sort=alt.EncodingSortField(field='count', op='sum', order='descending')
    )
)

# Bar Layer 
bars = base.mark_bar().encode(
    x=alt.X('count:Q', title='Number of Sightings'),
    # If the mouse is hovering, use Dark Blue; otherwise, use Light Blue
    color=alt.condition(
        hover_select,
        alt.value('#1f77b4'), # Dark Blue
        alt.value('#aec7e8')  # Light Blue
    ),
    tooltip=['shape', 'count']
).add_params(
    hover_select
)

# Text layer 
text = base.mark_text(align='left', baseline='middle', dx=3).encode(
    x='count:Q',
    text='count:Q'
)

# Combine 
ufo_chart_final = (bars + text).properties(
    title={
        "text": "Frequency of UFO Shapes Reported Globally",
        "subtitle": "Hover over a bar to highlight"
    },
    width=700,
    height=500
).configure_view(strokeWidth=0)

ufo_chart_final

alt.LayerChart(...)

In [7]:

ufo_chart_final.save('ufo_shapes.json')

## Chart 2

In [8]:
year_counts = df_ufo['year'].value_counts().reset_index()
year_counts.columns = ['year', 'count']
year_counts = year_counts.sort_values('year')

In [ ]:
brush = alt.selection_interval(encodings=['x'], name="brush")

# Line Chart
line = alt.Chart(year_counts).mark_line(color='#1f77b4', point=True).encode(
    x=alt.X('year:Q', title='Year of Sighting', axis=alt.Axis(format='d')),
    y=alt.Y('count:Q', title='Number of Sightings'),
    opacity=alt.condition(brush, alt.value(1), alt.value(0.2)),
    tooltip=['year:Q', 'count:Q']
)

# Text Layer 
stats_label = alt.Chart(year_counts).transform_filter(
    brush
).transform_aggregate(
    total='sum(count)',
    min_year='min(year)',
    max_year='max(year)'
).transform_calculate(
    label="datum.min_year + ' - ' + datum.max_year + ': ' + format(datum.total, ',d') + ' sightings'"
).mark_text(
    align='right', baseline='top', dx=-10, dy=10, 
    fontSize=14, fontWeight='bold', color='#1f77b4'
).encode(
    x=alt.value(210),
    y=alt.value(12),
    text='label:N'
)

# Combine
ufo_time_final = (line + stats_label).add_params(
    brush
).properties(
    title={
        "text": "UFO Sightings Trends (1906 - 2014)",
        "subtitle": "Drag to select a range"
    },
    width=700,
    height=400
).configure_view(strokeWidth=0)

# 6. Save 
ufo_time_final.save('ufo_time_dynamic.json')
ufo_time_final

alt.LayerChart(...)